# Table-Aware RAG vs Naive Chunking

`langchain_02.ipynb` uses `PyPDFLoader` + `RecursiveCharacterTextSplitter`, which flattens the formulary's drug tables into raw text and splits mid-row (a single drug's tier/limits can end up in a different chunk than its name).

This notebook builds a second pipeline using `pdfplumber`'s table extraction to keep each drug's full record (section, name, tier, requirements) as one chunk, and compares retrieval quality side-by-side against the naive approach on the same question.

In [1]:
# Load configuration from config.yaml
import yaml
from pathlib import Path

config_path = Path("../config/config.yaml")
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

pdf_dir = Path(f"../{config['paths']['raw_data']}")
pdf_path = next(pdf_dir.glob("*.pdf"))
print(f"✅ Config loaded. PDF: {pdf_path.name}")

✅ Config loaded. PDF: Oscar_4T_NC_STND_Member_Doc__January_2026__as_of_11182025.pdf


In [ ]:
# Shared embeddings model (used for both vector stores, for a fair comparison)
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name=config['embedding']['model_name'])
print(f"✅ Embeddings ready: {config['embedding']['model_name']}")

## Approach A - Naive (same as langchain_02.ipynb)

In [ ]:
# Load PDF pages and split with a fixed-size character splitter (loses table structure)
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

loader = PyPDFLoader(str(pdf_path))
all_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=config['chunking']['chunk_size'],
    chunk_overlap=config['chunking']['chunk_overlap'],
    length_function=len,
    is_separator_regex=False,
)
naive_chunks = text_splitter.split_documents(all_docs)

vectorstore_naive = FAISS.from_documents(naive_chunks, embeddings)
print(f"✅ Naive vector store: {len(naive_chunks)} chunks from {len(all_docs)} pages")

## Approach B - Table-aware, row-based chunking

Uses `pdfplumber.extract_tables()` to read each page's drug table as structured rows (`Drug Name`, `Drug Tier`, `Requirements/Limits`), tracks the current drug-class section header (e.g. "OPIOID ANALGESICS"), and turns each drug row into one self-contained `Document` - so a drug's name, tier, and limits can never be split across chunks.

In [3]:
# Extract one Document per drug row, keyed by its section/name/tier/requirements
import pdfplumber
from langchain_core.documents import Document

def extract_drug_rows(path):
    docs = []
    current_section = None  # persists across pages until the next section header

    with pdfplumber.open(path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            for table in page.extract_tables():
                for row in table:
                    cells = [(c or "").strip().replace("\n", " ") for c in row]
                    non_empty = [c for c in cells if c]

                    if not non_empty:
                        continue
                    if "Drug Name" in cells:  # repeated table header row
                        continue
                    if len(non_empty) == 1:  # section header, e.g. "OPIOID ANALGESICS"
                        current_section = non_empty[0]
                        continue
                    if len(cells) < 4:  # page footer / legend row
                        continue

                    _, drug_name, tier, requirements = cells[:4]
                    if not drug_name:
                        continue

                    content = (
                        f"Section: {current_section or 'N/A'}\n"
                        f"Drug: {drug_name}\n"
                        f"Tier: {tier or 'N/A'}\n"
                        f"Requirements/Limits: {requirements or 'None'}"
                    )
                    docs.append(Document(
                        page_content=content,
                        metadata={"section": current_section, "page": page_num, "drug": drug_name},
                    ))
    return docs

row_docs = extract_drug_rows(pdf_path)
print(f"✅ Extracted {len(row_docs)} drug-row documents")

✅ Extracted 2394 drug-row documents


In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name=config['embedding']['model_name'])
vectorstore_table = FAISS.from_documents(row_docs, embeddings)
print(f"✅ Table-aware vector store: {len(row_docs)} row-documents")

/tmp/ipykernel_19594/1835286237.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/tmp/ipykernel_19594/1835286237.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=config['embedding']['model_name'])
/home/bhargav/portfolio-project/healthins-enrollment-assistant-dev/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter an

✅ Table-aware vector store: 2394 row-documents


## LLM + retrieval test (table-aware only)

In [5]:
# Same local Ollama LLM and prompt used in langchain_02.ipynb
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOllama(
    model=config['llm']['model'],
    temperature=config['llm']['temperature'],
    num_predict=config['llm']['max_tokens'],
)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant answering questions about health insurance plans. Use only the provided context to answer."),
    ("user", "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"),
])

def ask(vectorstore, question, k=3):
    docs = vectorstore.similarity_search(question, k=k)
    context = "\n\n".join(d.page_content for d in docs)
    messages = prompt_template.format_messages(context=context, question=question)
    response = llm.invoke(messages)
    return response.content, docs

print(f"✅ LLM ready: {config['llm']['model']}")

✅ LLM ready: llama3.1


In [6]:
# Same question that exposed the naive approach's chunking problem, now at the DEFAULT top_k (no k=5 workaround)
question = (
    "What is the tier and quantity limit for acetaminophen with codeine "
    "solution 120-12 mg/5ml and tablet 300-15 mg?"
)
default_k = config['retrieval']['top_k']

print(f"=== TABLE-AWARE row chunking (top_k={default_k}) ===")
answer_table, docs_table = ask(vectorstore_table, question, k=default_k)
for d in docs_table:
    print("-", d.page_content.replace("\n", " "))
print("\nAnswer:", answer_table)

=== TABLE-AWARE row chunking (top_k=3) ===
- Section: OPIOID ANALGESICS Drug: acetaminophen w/ codeine soln 120-12 mg/5ml Tier: Tier 1 Requirements/Limits: QL (2700 ml every 30 days); Subject to initial 5- day limit for 19 and younger; 7-day initial limit for all other ages
- Section: OPIOID ANALGESICS Drug: acetaminophen w/ codeine tab 300-60 mg Tier: Tier 1 Requirements/Limits: QL (180 tabs every 30 days); Subject to initial 5- day limit for 19 and younger; 7-day initial limit for all other ages
- Section: OPIOID ANALGESICS Drug: acetaminophen w/ codeine tab 300-15 mg Tier: Tier 1 Requirements/Limits: QL (390 tabs every 30 days); Subject to initial 5- day limit for 19 and younger; 7-day initial limit for all other ages

Answer: For both the acetaminophen w/ codeine solution (120-12 mg/5ml) and the acetaminophen w/ codeine tablet (300-15 mg), the tier is Tier 1. 

The quantity limit (QL) for the solution is 2700 ml every 30 days. 

The quantity limit (QL) for the tablet is 390 tabs ev

## Validate the table-aware approach on unrelated drugs/sections

One passing question isn't proof - it's the same case the fix targets. Sampling a handful of
sections and running retrieval-only checks (no LLM, so this is fast) against DIFFERENT drugs
confirms the row-based chunking generalizes and isn't a fluke.

In [7]:
# Retrieval-only sanity check across several distinct sections (fast - no LLM calls)
import random

sections = sorted({d.metadata["section"] for d in row_docs if d.metadata["section"]})
print(f"Found {len(sections)} distinct sections, e.g.: {sections[:5]}")

random.seed(42)
sample_docs = random.sample(row_docs, 5)

for d in sample_docs:
    drug = d.metadata["drug"]
    query = f"What is the tier and requirements for {drug}?"
    results = vectorstore_table.similarity_search(query, k=default_k)
    top_drugs = [r.metadata["drug"] for r in results]
    hit = drug in top_drugs
    print(f"{'✅' if hit else '❌'} query for '{drug}' -> top match(es): {top_drugs}")

Found 220 distinct sections, e.g.: ['ACE INHIBITOR COMBINATIONS', 'ACE INHIBITORS', 'ACROMEGALY', 'ALCOHOL DETERRENTS', 'ALDOSTERONE RECEPTOR ANTAGONISTS']
✅ query for 'pemetrexed disodium solr 100mg, 500mg' -> top match(es): ['pemetrexed disodium solr 100mg, 500mg', 'decitabine solr 50mg', 'fludarabine phosphate soln 50mg/2ml; solr 50mg']
✅ query for 'oxycodone w/ acetaminophen tab 5-325 mg' -> top match(es): ['oxycodone w/ acetaminophen tab 2.5-325 mg', 'oxycodone w/ acetaminophen tab 5-325 mg', 'oxycodone w/ acetaminophen tab 7.5-325 mg']
✅ query for 'doxylamine succinate (sleep) tabs 25mg' -> top match(es): ['doxylamine succinate (sleep) tabs 25mg', 'doxepin hcl (sleep) tabs 3mg, 6mg', 'minoxidil tabs 2.5mg, 10mg']
✅ query for 'nortriptyline hcl caps 25mg' -> top match(es): ['nortriptyline hcl caps 75mg', 'nortriptyline hcl caps 25mg', 'nortriptyline hcl caps 10mg']
✅ query for 'EPIDIOLEX SOLN 100MG/ML' -> top match(es): ['EPIVIR HBV SOLN 5MG/ML', 'EPIDIOLEX SOLN 100MG/ML', 'epirub

In [8]:
# Full end-to-end LLM validation on one DIFFERENT section (not opioid analgesics)
non_opioid = [d for d in row_docs if d.metadata["section"] and "OPIOID" not in d.metadata["section"]]
random_drug_doc = random.choice(non_opioid)
print("Ground truth:\n", random_drug_doc.page_content)

validation_question = f"What tier and requirements apply to {random_drug_doc.metadata['drug']}?"
answer_val, docs_val = ask(vectorstore_table, validation_question, k=default_k)
print("\nRetrieved top match drug:", docs_val[0].metadata["drug"])
print("\nAnswer:", answer_val)

Ground truth:
 Section: ACE INHIBITORS
Drug: trandolapril tabs 1mg, 2mg, 4mg
Tier: Tier 1
Requirements/Limits: None

Retrieved top match drug: trandolapril tabs 1mg, 2mg, 4mg

Answer: Tier: Tier 1
Requirements/Limits: None
